In [1]:
import pandas as pd
import numpy as np
import feature_selection as fs
#import optuna
#from catboost import CatBoostClassifier
#from sklearn.metrics import accuracy_score
from sklearn.metrics import matthews_corrcoef
#from catboost import CatBoostClassifier
#from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
#from ydata_profiling import ProfileReport
import getting_threshold_genes as gtg
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
#from ydata_profiling import ProfileReport
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import matthews_corrcoef
#import matplotlib.pyplot as plt
#import seaborn as sns
#import shap
#import model as m
import catboost_model as miaw
#from catboost import Pool, cv
import pandas as pd
import numpy as np


from sklearn.metrics import matthews_corrcoef

from sklearn.model_selection import train_test_split
import getting_threshold_genes as gtg
#from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
#from ydata_profiling import ProfileReport
from sklearn.metrics import confusion_matrix
#from sklearn.metrics import classification_report
from sklearn.metrics import matthews_corrcoef
#from sklearn.metrics import roc_auc_score


from sklearn.metrics import make_scorer, matthews_corrcoef
#from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

from sklearn.metrics import make_scorer, matthews_corrcoef
#from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC



2024-11-11 17:52:08.957710: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-11 17:52:08.959720: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-11 17:52:08.986801: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-11 17:52:08.986828: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-11 17:52:08.987885: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Getting the data

In [2]:
filepath = '/home/karen/Documents/phd/Data/RNAseq_All_abundances_adjusted.csv'
eigth_bins_original = pd.read_csv(filepath)
eigth_bins_original=eigth_bins_original[eigth_bins_original["Experiment"]=="GSE167186"]
seq_metadata_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Data/All_rna_samples_metadata_edit.csv"
seq_metadata = pd.read_csv(seq_metadata_path)
rnaSeq_origial= pd.merge(eigth_bins_original, seq_metadata, on='Sample', how='inner')
rnaSeq_origial.head()

,Sample,ENSG00000000003.14,ENSG00000000005.5,ENSG00000000419.12,ENSG00000000457.13,ENSG00000000460.16,ENSG00000000938.12,ENSG00000000971.15,ENSG00000001036.13,ENSG00000001084.11,...,ENSG00000285476.1,ENSG00000285480.1,ENSG00000285491.1,ENSG00000285505.1,ENSG00000285508.1,ENSG00000285509.1,Experiment,Age,Status,Sex
0,SRR13758984,-0.018098,-0.039567,-0.004998,-0.033974,-0.035383,-0.037015,-0.021806,-0.029360,-0.023883,...,-0.038975,-0.037070,-0.038994,-0.038958,-0.027188,-0.038851,GSE167186,91.0,Sarcopenia,NaN
1,SRR13758985,-0.021714,-0.026491,-0.003711,-0.024397,-0.024751,-0.025862,-0.016819,-0.022847,-0.020888,...,-0.026331,-0.026158,-0.026407,-0.026438,-0.022984,-0.026243,GSE167186,86.0,Healthy,male
2,SRR13758986,-0.025526,-0.027849,-0.004777,-0.025392,-0.025384,-0.027021,-0.020357,-0.023013,-0.020575,...,-0.027645,-0.027292,-0.027715,-0.027739,-0.030522,-0.027553,GSE167186,69.0,Healthy,male
3,SRR13758987,-0.013807,-0.025538,-0.004040,-0.023699,-0.023917,-0.025049,-0.018966,-0.022372,-0.019911,...,-0.025410,-0.025362,-0.025489,-0.025526,-0.023519,-0.025324,GSE167186,83.0,Sarcopenia,NaN
4,SRR13758988,-0.029136,-0.028181,-0.003953,-0.024714,-0.026229,-0.027303,-0.017892,-0.022398,-0.020686,...,-0.027965,-0.027568,-0.028033,-0.028056,-0.024663,-0.027872,GSE167186,71.0,Healthy,NaN


In [3]:
len(rnaSeq_origial)

75

In [4]:
rnaSeq_origial['Sex'].value_counts()


Sex
male    29
Name: count, dtype: int64

Clean the data

In [5]:
x_labels= rnaSeq_origial["Status"].value_counts()
status_counts = rnaSeq_origial["Status"].value_counts()

rnaSeq_origial["Age_C"] = np.where(rnaSeq_origial["Age"] > 65, "Old", "Young")
rnaSeq_origial["Class"]=rnaSeq_origial["Status"].astype(str) +" "+ rnaSeq_origial["Age_C"].astype(str)

status_counts = rnaSeq_origial["Status"].value_counts()
status_counts = rnaSeq_origial["Class"].value_counts()
remove_youn = rnaSeq_origial[rnaSeq_origial["Age_C"]=="Old"]


In [6]:
len(remove_youn)

51

Get only te 997 genes from the union


In [7]:
ridge_7_selected_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/feature_importance_smote_ridge_7b_top_genes_elbow.csv"
ridge_7_selected = pd.read_csv(ridge_7_selected_path)
ridge_7_selected = ridge_7_selected["Ensembl"].tolist()

catboost_7_selected_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/feature_importance_smote_catboost_7b_gene_ranking_07.csv"
catboost_7_selected = pd.read_csv(catboost_7_selected_path)
catboost_7_selected = catboost_7_selected["Ensembl"].tolist()

catboost_40_selected_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/feature_importance_smote_catboost_7b_gene_ranking_40.csv"
catboost_40_selected = pd.read_csv(catboost_40_selected_path)
catboost_40_selected = catboost_40_selected["Ensembl"].tolist()
set_ridge_7 = set(ridge_7_selected)
set_catboost_7 = set(catboost_7_selected)
set_catboost_40 = set(catboost_40_selected)

union_genes = list(set(ridge_7_selected) | set(catboost_7_selected) | set(catboost_40_selected))
len(union_genes)

intersection_genes = list(set(ridge_7_selected) & set(catboost_7_selected) & set(catboost_40_selected))
len(intersection_genes)


24

In [8]:
# keep only genes in keepC AND the columns with name in union_genes
genes = intersection_genes
keepC = ["Sample", "Status", "Age","Sex", "Age_C", "Class", "Experiment"]
remove_youn_filtered = remove_youn[keepC + genes]
remove_youn_filtered.head()

,Sample,Status,Age,Sex,Age_C,Class,Experiment,ENSG00000151327.12,ENSG00000103145.10,ENSG00000171055.14,...,ENSG00000149925.18,ENSG00000175701.10,ENSG00000171033.12,ENSG00000094755.16,ENSG00000164879.6,ENSG00000179912.20,ENSG00000200755.1,ENSG00000222346.1,ENSG00000116288.12,ENSG00000164265.8
0,SRR13758984,Sarcopenia,91.0,NaN,Old,Sarcopenia Old,GSE167186,0.058762,0.200167,0.253797,...,3.246499,0.396713,0.339813,0.103801,0.030760,0.072127,-0.023797,-0.022316,0.112836,0.077697
1,SRR13758985,Healthy,86.0,male,Old,Healthy Old,GSE167186,0.031256,0.190451,0.098600,...,3.236078,0.397568,0.302488,0.106416,0.031764,0.038538,-0.021177,-0.019744,0.112992,0.080648
2,SRR13758986,Healthy,69.0,male,Old,Healthy Old,GSE167186,-0.001353,0.193975,0.069654,...,3.254627,0.396783,0.246701,0.108042,0.031547,0.039893,-0.021449,-0.020011,0.131070,0.076839
3,SRR13758987,Sarcopenia,83.0,NaN,Old,Sarcopenia Old,GSE167186,-0.000170,0.193951,0.074246,...,3.219239,0.398096,0.290123,0.106117,0.032724,0.026721,-0.020986,-0.019556,0.120036,0.079045
4,SRR13758988,Healthy,71.0,NaN,Old,Healthy Old,GSE167186,0.001323,0.200188,0.072093,...,3.257721,0.399042,0.295710,0.106485,0.031494,0.044497,-0.021515,-0.020076,0.118772,0.075892


In [9]:
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(remove_youn_filtered.drop(["Sample", "Status", "Age", "Sex", "Age_C", "Class", "Experiment"], axis=1), remove_youn_filtered["Status"], remove_youn_filtered["Age"], test_size=0.2, random_state=42)
len(seq_X_train), len(seq_X_test)

(40, 11)

In [10]:
train_df = pd.concat([seq_X_train, seq_y_train], axis=1)
train_df.columns=seq_X_train.columns.tolist()+["Age"]

smote = fs.get_feature_SMOTE(train_df, is_categorical=True, strategy={'Sarcopenia': 25, 'Healthy': 25} )


Class distribution before SMOTE: Counter({'Healthy': 23, 'Sarcopenia': 17})
Class distribution after SMOTE: Counter({'Sarcopenia': 25, 'Healthy': 25})


In [11]:
seq_X_train = smote[0]
seq_y_train = smote[1]
len(seq_y_train)

50

In [12]:
#noise = np.random.normal(0.01, 0.01, seq_X_train.shape)  # Generate random noise with mean 0 and standard deviation 1
#seq_X_train=seq_X_train+noise


In [13]:
scorer = make_scorer(matthews_corrcoef)
seq_y_train = seq_y_train.replace({"Healthy": 0, "Sarcopenia": 1})
seq_y_test = seq_y_test.replace({"Healthy": 0, "Sarcopenia": 1})
# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100, 1000, 10000],# 100000],  # Regularization parameter
    'gamma': [1, 0.1, 0.01, 0.001, 0.0001],# 0.00001],# 0.000001],  # Kernel coefficient for 'rbf' and 'poly'
    'kernel': [ 'poly', 'sigmoid'],  # Kernel type
}

# Create SVM model
svm_model = SVC()

# Perform grid search with cross-validation
grid_search = GridSearchCV(svm_model, param_grid, cv=10, scoring=scorer)
grid_search.fit(seq_X_train, seq_y_train)

# Get best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_



/tmp/ipykernel_13792/1735117252.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  seq_y_train = seq_y_train.replace({"Healthy": 0, "Sarcopenia": 1})
/tmp/ipykernel_13792/1735117252.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  seq_y_test = seq_y_test.replace({"Healthy": 0, "Sarcopenia": 1})


In [14]:

print("Best Parameters:", best_params)
print("Best Score (mcc):", best_score)

Best Parameters: {'C': 1000, 'gamma': 1, 'kernel': 'poly'}
Best Score (mcc): 0.5316496580927726


In [16]:
best_score

0.5316496580927726

In [17]:

# Use best model for prediction
best_svm_model = grid_search.best_estimator_
y_pred = best_svm_model.predict(seq_X_test)
y_pred

array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1])

In [18]:
conf_matrix_train = confusion_matrix(seq_y_test, y_pred)
print(conf_matrix_train)

[[4 1]
 [4 2]]


In [19]:
## Get the classification report
class_report = classification_report(seq_y_test, y_pred)
print(class_report)

              precision    recall  f1-score   support

           0       0.50      0.80      0.62         5
           1       0.67      0.33      0.44         6

    accuracy                           0.55        11
   macro avg       0.58      0.57      0.53        11
weighted avg       0.59      0.55      0.52        11

